# Clean US Tariff data from Federal Registry

What:
- extract tariff announcement "signals" for selected sectors: ["aluminum", "steel"]

## Setup

In [2]:
import pandas as pd
from pathlib import Path

## Configs

In [86]:
RAW_DATA_PATH = Path("../data/raw/fedregister/fedregister_raw.parquet")
CLEAN_DATA_PATH = Path("../data/tidy/fedregister_clean.parquet")

## Main

### Load

In [4]:
df_raw = pd.read_parquet(RAW_DATA_PATH)
print(f"Raw data loaded")

Raw data loaded


### Data Inspection

In [5]:
print(f"Shape of the raw data set: {df_raw.shape}")

Shape of the raw data set: (36686, 10)


In [6]:
df_raw.head()

,title,type,abstract,document_number,html_url,pdf_url,public_inspection_pdf_url,publication_date,agencies,excerpts
0,"Seamless Carbon and Alloy Steel Standard, Line...",Notice,The Commission hereby gives notice of the sche...,2020-28986,https://www.federalregister.gov/documents/2020...,https://www.govinfo.gov/content/pkg/FR-2020-12...,https://public-inspection.federalregister.gov/...,2020-12-31,"[{'id': 262.0, 'json_url': 'https://www.federa...",731-TA-1529-1532 (Final) pursuant to the <span...
1,Final Results of Expedited Sunset Review of Co...,Notice,"As a result of this second sunset review, the ...",2020-28984,https://www.federalregister.gov/documents/2020...,https://www.govinfo.gov/content/pkg/FR-2020-12...,https://public-inspection.federalregister.gov/...,2020-12-31,"[{'id': 54.0, 'json_url': 'https://www.federal...","Department of Commerce, 1401 Constitution Aven..."
2,Diamond Sawblades and Parts Thereof From the P...,Notice,"On November 20, 2020, the Department of Commer...",2020-28980,https://www.federalregister.gov/documents/2020...,https://www.govinfo.gov/content/pkg/FR-2020-12...,https://public-inspection.federalregister.gov/...,2020-12-31,"[{'id': 54.0, 'json_url': 'https://www.federal...",sawblades and/or diamond segment(s) with diamo...
3,Prestressed Concrete Steel Wire Strand From th...,Notice,The Department of Commerce (Commerce) finds th...,2020-28979,https://www.federalregister.gov/documents/2020...,https://www.govinfo.gov/content/pkg/FR-2020-12...,https://public-inspection.federalregister.gov/...,2020-12-31,"[{'id': 54.0, 'json_url': 'https://www.federal...","INFORMATION: \n Background \n \n On June 29, 2..."
4,Paulsboro Refining Company LLC; Supplemental N...,Notice,None,2020-28977,https://www.federalregister.gov/documents/2020...,https://www.govinfo.gov/content/pkg/FR-2020-12...,https://public-inspection.federalregister.gov/...,2020-12-31,"[{'id': 136.0, 'json_url': 'https://www.federa...",This is a supplemental notice in the above-ref...


In [7]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36686 entries, 0 to 36685
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   title                      36686 non-null  object
 1   type                       36686 non-null  object
 2   abstract                   25593 non-null  object
 3   document_number            36686 non-null  object
 4   html_url                   36686 non-null  object
 5   pdf_url                    36686 non-null  object
 6   public_inspection_pdf_url  36646 non-null  object
 7   publication_date           36686 non-null  object
 8   agencies                   36686 non-null  object
 9   excerpts                   36686 non-null  object
dtypes: object(10)
memory usage: 2.8+ MB


In [8]:
# df_raw.describe(include="all")

In [9]:
# move to Notes
# for idx, row in df_raw["abstract"].head(10).items():
#     if not isinstance(row, str):
#         print(f"Row {idx} is not a string: {row}")
#         print(row)

### Data type conversion

In [10]:
print(df_raw["publication_date"].dtype)

object


`publication_data` is `object` but should be `datetime`

In [11]:
print(f"Investigate 'publication_date' column before type change:")
display(df_raw["publication_date"])

print(f"type:")
print(df_raw["publication_date"].dtype)

Investigate 'publication_date' column before type change:


0        2020-12-31
1        2020-12-31
2        2020-12-31
3        2020-12-31
4        2020-12-31
            ...    
36681    2024-01-02
36682    2024-01-02
36683    2024-01-02
36684    2024-01-02
36685    2024-01-02
Name: publication_date, Length: 36686, dtype: object

type:
object


In [12]:
# Change 'publication_date' to datetime
df_raw["publication_date"] = pd.to_datetime(
    df_raw["publication_date"], errors="coerce"
)

In [13]:
print(f"Investigate 'publication_date' column after type change:")
display(df_raw["publication_date"])

print(f"type:")
print(df_raw["publication_date"].dtype)

Investigate 'publication_date' column after type change:


0       2020-12-31
1       2020-12-31
2       2020-12-31
3       2020-12-31
4       2020-12-31
           ...    
36681   2024-01-02
36682   2024-01-02
36683   2024-01-02
36684   2024-01-02
36685   2024-01-02
Name: publication_date, Length: 36686, dtype: datetime64[ns]

type:
datetime64[ns]


### Duplicates

In [14]:
df_raw.iloc[0]

title                        Seamless Carbon and Alloy Steel Standard, Line...
type                                                                    Notice
abstract                     The Commission hereby gives notice of the sche...
document_number                                                     2020-28986
html_url                     https://www.federalregister.gov/documents/2020...
pdf_url                      https://www.govinfo.gov/content/pkg/FR-2020-12...
public_inspection_pdf_url    https://public-inspection.federalregister.gov/...
publication_date                                           2020-12-31 00:00:00
agencies                     [{'id': 262.0, 'json_url': 'https://www.federa...
excerpts                     731-TA-1529-1532 (Final) pursuant to the <span...
Name: 0, dtype: object

I chose the `document_number` as a deduplication key

In [36]:
deduplication_keys = ["document_number", "abstract"]

In [101]:
df_dedup = df_raw.drop_duplicates(subset=deduplication_keys).copy()
percentage_unique = len(df_dedup) / len(df_raw) * 100

print(f"Unique documents after deduplication: {len(df_dedup):,}")
print(f"Unique documents before deduplication: {len(df_raw):,}")
print(
    f"Percentage of unique documents after deduplication: {percentage_unique:.2f}%"
)

Unique documents after deduplication: 32,764
Unique documents before deduplication: 36,686
Percentage of unique documents after deduplication: 89.31%


### Inspect missing values

In [102]:
df_raw.isnull().sum()

title                            0
type                             0
abstract                     11093
document_number                  0
html_url                         0
pdf_url                          0
public_inspection_pdf_url       40
publication_date                 0
agencies                         0
excerpts                         0
dtype: int64

Of the two columns with missing values, only `abstract` seems to be impactful

<!-- To decide on the next step, it needs investigating -->

These missing values will be dealt with directly in feature extraction section

### Extract Features

Investigate 

In [103]:
# List column names
list(df_dedup.columns)

['title',
 'type',
 'abstract',
 'document_number',
 'html_url',
 'pdf_url',
 'public_inspection_pdf_url',
 'publication_date',
 'agencies',
 'excerpts']

In [104]:
# List content of each column in the df's first row
for idx, row in df_dedup.iloc[0].items():
    print(f"{idx}:\n\n{row}\n{'-'*120}\n\n")

title:

Seamless Carbon and Alloy Steel Standard, Line, and Pressure Pipe From Czechia, Korea, Russia, and Ukraine; Scheduling of the Final Phase of Countervailing Duty and Antidumping Duty Investigations
------------------------------------------------------------------------------------------------------------------------


type:

Notice
------------------------------------------------------------------------------------------------------------------------


abstract:

The Commission hereby gives notice of the scheduling of the final phase of antidumping and countervailing duty investigation Nos. 701-TA-654-655 and 731-TA-1529-1532 (Final) pursuant to the Tariff Act of 1930 ("the Act") to determine whether an industry in the United States is materially injured or threatened with material injury, or the establishment of an industry in the United States is materially retarded, by reason of imports of seamless carbon and alloy steel standard, line, and pressure pipe from Czechia, Korea,

Column `abstract` looks like a promising candidate to filter for desired sectors

#### 1. Basic cleaning

- Lowercase and strip the `abstract` and `title` columns to standardize text for matching
- Missing values (e.g. `None`) are handled - converted to empty strings

Verify how many rows have an `abstract` or `title` that is not a `str` (e.g. `None`, `NaN`, etc.)

In [106]:
non_str_abstracts = df_dedup[
    df_dedup["abstract"].apply(lambda x: not isinstance(x, str))
].shape[0]
print(
    f"Rows with non-string abstracts: {non_str_abstracts:,} ({non_str_abstracts / len(df_dedup) * 100:.2f}%)"
)

Rows with non-string abstracts: 10,725 (32.73%)


In [108]:
non_str_titles = df_dedup[
    df_dedup["title"].apply(lambda x: not isinstance(x, str))
].shape[0]
print(
    f"Rows with non-string titles: {non_str_titles:,} ({non_str_titles / len(df_dedup) * 100:.2f}%)"
)

Rows with non-string titles: 0 (0.00%)


Clean function

In [109]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    return text.lower().strip().replace("\n", " ").replace("\r", " ")

Apply cleaning

In [110]:
df_clean = df_dedup.copy()
df_clean["abstract"] = df_clean["abstract"].apply(clean_text)
df_clean["title"] = df_clean["title"].apply(clean_text)

#### 2. Filter for sector-specific keywords

- Mark rows that mention target sectors in the `abstract`
- Narrows down the dataset to potentially relevant tariff-related actions

In [133]:
SECTOR_KEYWORDS = {
    "aluminum": ["aluminum", "aluminium"],
    "steel": ["steel"],
    "semiconductor": ["semiconductor", "chip", "integrated circuit", "ic"],
}

In [134]:
for sector, sector_keywords in SECTOR_KEYWORDS.items():
    df_clean[f"{sector}_match"] = df_clean["abstract"].apply(
        lambda x: any(keyword in x for keyword in sector_keywords)
    )

In [152]:
for sector, count in df_clean.filter(regex="_match").sum().items():
    print(f"{sector}: {count:,} ({count / len(df_clean) * 100:.2f}%)")

aluminum_match: 278 (0.85%)
steel_match: 1,226 (3.74%)
semiconductor_match: 20,208 (61.68%)


#### 3. Explore sector flags to validate them 

Aluminum

In [162]:
df_aluminum = df_clean[df_clean["aluminum_match"]]
display(df_aluminum.sample(50))

,title,type,abstract,document_number,html_url,pdf_url,public_inspection_pdf_url,publication_date,agencies,excerpts,aluminum_match,steel_match,semiconductor_match
4098,common alloy aluminum sheet from india: final ...,Notice,the department of commerce (commerce) determin...,2021-04728,https://www.federalregister.gov/documents/2021...,https://www.govinfo.gov/content/pkg/FR-2021-03...,https://public-inspection.federalregister.gov/...,2021-03-08,"[{'id': 54.0, 'json_url': 'https://www.federal...","482-5305, respectively. \n \n \n \n SUPPLEMENT...",True,False,False
3748,certain aluminum foil from the sultanate of om...,Notice,the department of commerce (commerce) prelimin...,2021-09320,https://www.federalregister.gov/documents/2021...,https://www.govinfo.gov/content/pkg/FR-2021-05...,https://public-inspection.federalregister.gov/...,2021-05-04,"[{'id': 54.0, 'json_url': 'https://www.federal...","Trade Administration, U.S. Department of Comme...",True,False,False
4092,common alloy aluminum sheet from south africa:...,Notice,the department of commerce (commerce) determin...,2021-04734,https://www.federalregister.gov/documents/2021...,https://www.govinfo.gov/content/pkg/FR-2021-03...,https://public-inspection.federalregister.gov/...,2021-03-08,"[{'id': 54.0, 'json_url': 'https://www.federal...","determination in this investigation. However, ...",True,False,True
6461,certain aluminum foil from the people's republ...,Notice,the department of commerce (commerce) publishe...,2022-00282,https://www.federalregister.gov/documents/2022...,https://www.govinfo.gov/content/pkg/FR-2022-01...,https://public-inspection.federalregister.gov/...,2022-01-11,"[{'id': 54.0, 'json_url': 'https://www.federal...","Materials Joint-Stock Co., Ltd.; Luoyang Longd...",True,False,True
5134,common alloy aluminum sheet from the people's ...,Notice,the u.s. department of commerce (commerce) det...,2022-19342,https://www.federalregister.gov/documents/2022...,https://www.govinfo.gov/content/pkg/FR-2022-09...,https://public-inspection.federalregister.gov/...,2022-09-08,"[{'id': 54.0, 'json_url': 'https://www.federal...","for these final results until August 31, 2022....",True,False,True
3745,certain aluminum foil from the republic of tur...,Notice,the department of commerce (commerce) prelimin...,2021-09323,https://www.federalregister.gov/documents/2021...,https://www.govinfo.gov/content/pkg/FR-2021-05...,https://public-inspection.federalregister.gov/...,2021-05-04,"[{'id': 54.0, 'json_url': 'https://www.federal...","International Trade Administration, U.S. Depar...",True,False,True
6806,common alloy aluminum sheet from turkey: final...,Notice,the u.s. department of commerce (commerce) det...,2023-24886,https://www.federalregister.gov/documents/2023...,https://www.govinfo.gov/content/pkg/FR-2023-11...,https://public-inspection.federalregister.gov/...,2023-11-13,"[{'id': 54.0, 'json_url': 'https://www.federal...","publication of the \n Preliminary Results, \n ...",True,False,True
3746,certain aluminum foil from russia: preliminary...,Notice,the department of commerce (commerce) prelimin...,2021-09322,https://www.federalregister.gov/documents/2021...,https://www.govinfo.gov/content/pkg/FR-2021-05...,https://public-inspection.federalregister.gov/...,2021-05-04,"[{'id': 54.0, 'json_url': 'https://www.federal...","International Trade Administration, U.S. Depar...",True,False,False
2843,certain aluminum foil from the republic of tur...,Notice,the department of commerce (commerce) determin...,2021-20535,https://www.federalregister.gov/documents/2021...,https://www.govinfo.gov/content/pkg/FR-2021-09...,https://public-inspection.federalregister.gov/...,2021-09-23,"[{'id': 54.0, 'json_url': 'https://www.federal...","(202) 482-3878, respectively. \n \n \n \n SUPP...",True,False,True
10035,aluminum extrusions from the socialist republi...,Notice,the u.s. department of commerce (commerce) pre...,2024-09932,https://www.federalregister.gov/documents/2024...,https://www.govinfo.gov/c

In [ ]:
for row in (
    df_aluminum[["title", "abstract", "excerpts"]]
    .sample(50, random_state=42)
    .iterrows()
):
    idx, row_data = row
    print("=" * 120 + "\n")
    print(f"Row {idx}:\n")
    print("-" * 120 + "\n")
    print(f"Title: {row_data['title']}\n")
    print("-" * 120 + "\n")
    print(f"Abstract: {row_data['abstract']}\n")
    print("-" * 120 + "\n")
    print(f"Excerpts: {row_data['excerpts']}\n")
    print("=" * 120 + "\n")


Row 1189:

------------------------------------------------------------------------------------------------------------------------

Title: certain aluminum foil from the people's republic of china: preliminary results of the countervailing duty administrative review and rescission of review, in part; 2017-2018

------------------------------------------------------------------------------------------------------------------------

Abstract: the department of commerce (commerce) preliminarily determines that countervailable subsidies are being provided to producers and exporters of certain aluminum foil (aluminum foil) from the people's republic of china (china). the period of review is august 14, 2017 through december 31, 2018.

------------------------------------------------------------------------------------------------------------------------

Excerpts: Memorandum).
 
 
 Scope of the Order 
 
 The product covered by the 
 Order 
 is aluminum foil from China. For a complete descr

Conclusion - despite fetching the date with proper terms (`"conditions[term]": term`), there are still irrelevant entries, like form EPA or FAA that don't normally impose tariffs. Thus, fetching has been rerun, restricting the date only to proper agencies, capable of actually imposing tariffs (ids taken from https://www.federalregister.gov/api/v1/agencies): 
- White House
  - `id: 2` (item 1)
- Department of Commerce
  - `id: 54` (item 52)
- United States Trade Representative (USTR)
  - `id: 491` (item 427)

Fetch was the run with:

```python
params = {
    "conditions[term]": term,
    "conditions[publication_date][gte]": f"{year}-01-01",
    "conditions[publication_date][lte]": f"{year}-12-31",
    "per_page": 1000,
    "page": page,
    "order": "newest",
    "conditions[agencies]": [2, 54, 491], # White House, Commerce, USTR
}
```

instead of original:
```python
params = {
    "conditions[term]": term,
    "conditions[publication_date][gte]": f"{year}-01-01",
    "conditions[publication_date][lte]": f"{year}-12-31",
    "per_page": 1000,
    "page": page,
    "order": "newest",
}
```

### Save

In [ ]:
CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.to_parquet(CLEAN_DATA_PATH, index=False)
print(f"Cleaned data saved to: {CLEAN_DATA_PATH}")

Cleaned data saved to: ../data/tidy/fedregister_clean.parquet


## Leftover code

### Search for `<span.?>` tags in `excerpt`

In [104]:
# def find_span_tags(text):
#     if not isinstance(text, str):
#         return []
#     return re.findall(r"<span.*?>.*?</span>", text)


# span_hits = df_raw[df_raw["excerpts"].notnull()].copy()
# span_hits["span_matches"] = span_hits["excerpts"].apply(find_span_tags)

# # Show the first actual match
# span_hits = span_hits[span_hits["span_matches"].apply(len) > 0]
# span_hits.iloc[0]["span_matches"]

### Sample

In [ ]:
# sample = df_raw[df_raw["abstract"].notna()].sample(1, random_state=42)
# for item in sample.iloc[0]:
#     print(f"{item}\n")

### Missing

In [84]:
# df_raw[df_raw["abstract"].isna()].shape[0]